# CharNet Detection Visualiser

Load document images alongside their CharNet JSON outputs and visualise
**word** and **character** bounding boxes interactively.

- **Word boxes** are drawn as coloured oriented polygons with the recognised text as a label.
- **Character boxes** inside each word share the same colour and show the top-1 predicted character.

---

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────
# Point these to any input/output root pair produced by run_charnet.py.

IMAGE_ROOT  = r"../data/corpus-1/imgs"   # mirrors the --input_root used
JSON_ROOT   = r"../data/corpus-1/charnet"   # mirrors the --output_root used  (adjust if different)

# Which document to inspect (subfolder name under IMAGE_ROOT / JSON_ROOT)
DOCUMENT    = "BNE_1001_615_T-55281-18"

# Which page (without extension)
PAGE        = "page_10"

In [ ]:
import json
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

# nice defaults
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 8,
    "axes.titlesize": 11,
})

In [ ]:
# ── Load image & detections ──────────────────────────────────────────

img_path  = Path(IMAGE_ROOT) / DOCUMENT / f"{PAGE}.png"
json_path = Path(JSON_ROOT)  / DOCUMENT / f"{PAGE}.json"

assert img_path.exists(),  f"Image not found: {img_path}"
assert json_path.exists(), f"JSON not found: {json_path}"

img_bgr = cv2.imread(str(img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

with open(json_path) as f:
    words = json.load(f)

print(f"Image size : {img_rgb.shape[1]}×{img_rgb.shape[0]}")
print(f"Words found: {len(words)}")
print(f"Chars total: {sum(len(w.get('chars', [])) for w in words)}")

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────

def polygon_from_flat(coords):
    """Convert [x1,y1,x2,y2,x3,y3,x4,y4] → (4,2) array."""
    return np.array(coords, dtype=float).reshape(4, 2)


def make_colour_map(n, cmap_name="hsv"):
    """Return *n* distinct RGBA colours from a matplotlib colourmap."""
    cmap = plt.cm.get_cmap(cmap_name, max(n, 1))
    return [cmap(i) for i in range(n)]


def top1_label(labels_dict):
    """Return the character with the highest score."""
    return max(labels_dict, key=labels_dict.get)

In [ ]:
# ── Full-page plot: words + chars ────────────────────────────────────

colours = make_colour_map(len(words), cmap_name="tab20")

fig, ax = plt.subplots(1, 1, figsize=(18, 24))
ax.imshow(img_rgb)
ax.set_axis_off()
ax.set_title(f"{DOCUMENT} / {PAGE}  —  {len(words)} words", fontweight="bold")

for idx, word in enumerate(words):
    colour = colours[idx % len(colours)]

    # ── word polygon ──
    if "polygon" in word:
        poly = polygon_from_flat(word["polygon"])
        patch = MplPolygon(poly, closed=True,
                           linewidth=2, edgecolor=colour,
                           facecolor=(*colour[:3], 0.08))
        ax.add_patch(patch)

        # word text label
        cx, cy = poly.mean(axis=0)
        text_str = word.get("text", "")
        score = word.get("text_score", 0)
        ax.text(cx, cy, f"{text_str}",
                fontsize=7, fontweight="bold",
                color="white",
                bbox=dict(boxstyle="round,pad=0.15",
                          facecolor=colour, alpha=0.85,
                          edgecolor="none"),
                ha="center", va="center", clip_on=True)

    # ── character polygons ──
    for char_det in word.get("chars", []):
        if "polygon" not in char_det:
            continue
        cpoly = polygon_from_flat(char_det["polygon"])
        cpatch = MplPolygon(cpoly, closed=True,
                            linewidth=0.8, edgecolor=colour,
                            facecolor="none", linestyle="--")
        ax.add_patch(cpatch)

        # character label
        ccx, ccy = cpoly.mean(axis=0)
        lbl = top1_label(char_det["labels"])
        ax.text(ccx, ccy - 6, lbl,
                fontsize=5, color=colour, fontweight="bold",
                ha="center", va="bottom", clip_on=True)

fig.tight_layout()
plt.show()

---
## Zoom into a region

Pick a rectangular crop (in pixels) to zoom into a specific area and see
the character-level detail more clearly.

In [ ]:
# ── Crop coordinates (adjust as needed) ──────────────────────────────
X_MIN, X_MAX = 0, 1500
Y_MIN, Y_MAX = 0, 1200

In [ ]:
# ── Zoomed plot ──────────────────────────────────────────────────────

fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.imshow(img_rgb)
ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(Y_MAX, Y_MIN)  # y-axis is inverted in image coords
ax.set_axis_off()
ax.set_title(f"Zoom: ({X_MIN},{Y_MIN})–({X_MAX},{Y_MAX})", fontweight="bold")

for idx, word in enumerate(words):
    colour = colours[idx % len(colours)]

    if "polygon" not in word:
        continue
    poly = polygon_from_flat(word["polygon"])

    # skip words completely outside the crop
    if (poly[:, 0].max() < X_MIN or poly[:, 0].min() > X_MAX or
            poly[:, 1].max() < Y_MIN or poly[:, 1].min() > Y_MAX):
        continue

    patch = MplPolygon(poly, closed=True,
                       linewidth=2.5, edgecolor=colour,
                       facecolor=(*colour[:3], 0.10))
    ax.add_patch(patch)

    cx, cy = poly.mean(axis=0)
    text_str = word.get("text", "")
    score = word.get("text_score", 0)
    ax.text(cx, cy, f"{text_str}  ({score:.0%})",
            fontsize=9, fontweight="bold", color="white",
            bbox=dict(boxstyle="round,pad=0.2",
                      facecolor=colour, alpha=0.85,
                      edgecolor="none"),
            ha="center", va="center", clip_on=True)

    for char_det in word.get("chars", []):
        if "polygon" not in char_det:
            continue
        cpoly = polygon_from_flat(char_det["polygon"])
        cpatch = MplPolygon(cpoly, closed=True,
                            linewidth=1.2, edgecolor=colour,
                            facecolor=(*colour[:3], 0.06),
                            linestyle="--")
        ax.add_patch(cpatch)

        ccx, ccy = cpoly.mean(axis=0)
        lbl = top1_label(char_det["labels"])
        ax.text(ccx, ccy - 8, lbl,
                fontsize=8, color=colour, fontweight="bold",
                ha="center", va="bottom", clip_on=True)

fig.tight_layout()
plt.show()

---
## Words table

A summary of every detected word with its recognition confidence.

In [ ]:
import pandas as pd

rows = []
for i, w in enumerate(words):
    rows.append({
        "#": i,
        "text": w.get("text", ""),
        "text_score": round(w.get("text_score", 0), 3),
        "word_bbox_score": round(w.get("word_bbox_score", 0), 3),
        "n_chars": len(w.get("chars", [])),
        "tblr": w.get("tblr"),
    })

df = pd.DataFrame(rows)

# Style: colour-code the text score
styled = (
    df.style
    .background_gradient(subset=["text_score"], cmap="RdYlGn", vmin=0.5, vmax=1.0)
    .background_gradient(subset=["word_bbox_score"], cmap="Blues", vmin=0, vmax=2.0)
    .format({"text_score": "{:.3f}", "word_bbox_score": "{:.3f}"})
)
styled

---
## Browse all pages in a document

Run the cell below to list every page that has both an image **and** a
JSON file.  Then change `PAGE` in the first cell and re-run.

In [ ]:
img_dir  = Path(IMAGE_ROOT) / DOCUMENT
json_dir = Path(JSON_ROOT)  / DOCUMENT

available = sorted(
    p.stem for p in img_dir.glob("*.png")
    if (json_dir / f"{p.stem}.json").exists()
)
print(f"{len(available)} pages with detections in '{DOCUMENT}':")
for name in available:
    n_words = len(json.load(open(json_dir / f"{name}.json")))
    print(f"  {name:20s}  ({n_words:3d} words)")